In [110]:
import numpy as np
import os
import sys
import wfdb
from utils_paf import qrs_detect, comp_cosEn, save_dict
import matplotlib.pyplot as plt
import pandas as pd
from ecg_preprocessing import preprocess
from scipy.signal import resample
import csv
import math

In [112]:
def load_data(sample_path):
    sig, fields = wfdb.rdsamp(sample_path)
    length = len(sig)
    fs = fields['fs']

    return sig, length, fs

In [114]:
def ngrams_rr(data, length):
    grams = []
    for i in range(0, length-12, 12):
        grams.append(data[i: i+12])
    return grams

In [116]:
def challenge_entry(sample_path):
    """
    This is a baseline method.
    """

    sig, _, fs = load_data(sample_path)
    sig = sig[:, 1]
    end_points = []

    r_peaks = qrs_detect(sig, fs=200)
    print(r_peaks)
    rr_seq = np.diff(r_peaks) / fs
    len_rr = len(rr_seq)

    rr_seq_slice = ngrams_rr(rr_seq, len_rr)
    is_af = []
    for rr_period in rr_seq_slice:
        cos_en, _ = comp_cosEn(rr_period)
        if cos_en <= -1.4:
            is_af.append(0)
        else:
            is_af.append(1)
    is_af = np.array([[j] * 12 for j in is_af]).flatten()
    rr_seq_last = rr_seq[-12: ]
    cos_en, _ = comp_cosEn(rr_seq_last)
    if cos_en <= -1.4:
        is_af_last = 0
    else:
        is_af_last = 1
    
    len_rr_remain = len_rr - int(12*len(rr_seq_slice))
    is_af = np.concatenate((is_af, np.array([is_af_last] * len_rr_remain).flatten()), axis=0)

    if np.sum(is_af) == len(is_af):
        end_points.append([0, len(sig)-1])
    elif np.sum(is_af) != 0:
        state_diff = np.diff(is_af)
        start_r = np.where(state_diff==1)[0] + 1
        end_r = np.where(state_diff==-1)[0] + 1

        if is_af[0] == 1:
            start_r = np.insert(start_r, 0, 0)
        if is_af[-1] == 1:
            end_r = np.insert(end_r, len(end_r), len(is_af)-1)
        start_r = np.expand_dims(start_r, -1)
        end_r = np.expand_dims(end_r, -1)
        start_end = np.concatenate((r_peaks[start_r], r_peaks[end_r]), axis=-1).tolist()
        end_points.extend(start_end)
        
    pred_dcit = {'predict_endpoints': end_points}
    
    return pred_dcit

In [118]:
def buffer(data,duration,dataOverlap):
    numberOfSegments = int(math.ceil((len(data)-dataOverlap)/(duration-dataOverlap)))
    #print(data.shape)
    tempBuf = [data[i:i+duration] for i in range(0,len(data),(duration-int(dataOverlap)))]
    tempBuf[numberOfSegments-1] = np.pad(tempBuf[numberOfSegments-1],(0,duration-tempBuf[numberOfSegments-1].shape[0]),'constant')
    tempBuf2 = np.vstack(tempBuf[0:numberOfSegments])
    return tempBuf2

In [120]:
class RefInfo():
    def __init__(self, sample_path):
        self.sample_path = sample_path
        self.fs, self.len_sig, self.beat_loc, self.af_starts, self.af_ends, self.class_true = self._load_ref()
        self.endpoints_true = np.dstack((self.af_starts, self.af_ends))[0, :, :]
        # self.endpoints_true = np.concatenate((self.af_starts, self.af_ends), axis=-1)

        if self.class_true == 1 or self.class_true == 2:
            self.onset_score_range, self.offset_score_range = self._gen_endpoint_score_range()
        else:
            self.onset_score_range, self.offset_score_range = None, None

    def _load_ref(self):
        sig, fields = wfdb.rdsamp(self.sample_path)
        ann_ref = wfdb.rdann(self.sample_path, 'atr')

        fs = fields['fs']
        length = len(sig)
        sample_descrip = fields['comments']

        beat_loc = np.array(ann_ref.sample) # r-peak locations
        ann_note = np.array(ann_ref.aux_note) # rhythm change flag

        af_start_scripts = np.where((ann_note=='(AFIB') | (ann_note=='(AFL'))[0]
        af_end_scripts = np.where(ann_note=='(N')[0]

        if 'non atrial fibrillation' in sample_descrip:
            class_true = 0
        elif 'persistent atrial fibrillation' in sample_descrip:
            class_true = 1
        elif 'paroxysmal atrial fibrillation' in sample_descrip:
            class_true = 2
        else:
            print('Error: the recording is out of range!')

            return -1

        return fs, length, beat_loc, af_start_scripts, af_end_scripts, class_true
    
    def _gen_endpoint_score_range(self):
        """

        """
        onset_range = np.zeros((self.len_sig, ),dtype=float)
        offset_range = np.zeros((self.len_sig, ),dtype=float)
        for i, af_start in enumerate(self.af_starts):
            if self.class_true == 2:
                if max(af_start-1, 0) == 0:
                    onset_range[: self.beat_loc[af_start+2]] += 1
                elif max(af_start-2, 0) == 0:
                    onset_range[self.beat_loc[af_start-1]: self.beat_loc[af_start+2]] += 1
                    onset_range[: self.beat_loc[af_start-1]] += .5
                else:
                    onset_range[self.beat_loc[af_start-1]: self.beat_loc[af_start+2]] += 1
                    onset_range[self.beat_loc[af_start-2]: self.beat_loc[af_start-1]] += .5
                onset_range[self.beat_loc[af_start+2]: self.beat_loc[af_start+3]] += .5
            elif self.class_true == 1:
                onset_range[: self.beat_loc[af_start+2]] += 1
                onset_range[self.beat_loc[af_start+2]: self.beat_loc[af_start+3]] += .5
        for i, af_end in enumerate(self.af_ends):
            if self.class_true == 2:
                if min(af_end+1, len(self.beat_loc)-1) == len(self.beat_loc)-1:
                    offset_range[self.beat_loc[af_end-2]: ] += 1
                elif min(af_end+2, len(self.beat_loc)-1) == len(self.beat_loc)-1:
                    offset_range[self.beat_loc[af_end-2]: self.beat_loc[af_end+1]] += 1
                    offset_range[self.beat_loc[af_end+1]: ] += 0.5
                else:
                    offset_range[self.beat_loc[af_end-2]: self.beat_loc[af_end+1]] += 1
                    offset_range[self.beat_loc[af_end+1]: min(self.beat_loc[af_end+2], self.len_sig-1)] += .5
                offset_range[self.beat_loc[af_end-3]: self.beat_loc[af_end-2]] += .5 
            elif self.class_true == 1:
                offset_range[self.beat_loc[af_end-2]: ] += 1
                offset_range[self.beat_loc[af_end-3]: self.beat_loc[af_end-2]] += .5
        
        return onset_range, offset_range

In [122]:
DATA_PATH = 'D:\\research papers\\Digital Twinning\\paroxysmal-atrial-fibrillation-events-detection-from-dynamic-ecg-recordings-the-4th-china-physiological-signal-challenge-2021-1.0.0\\'


In [124]:
test_set = open(os.path.join(DATA_PATH, 'RECORDS'), 'r').read().splitlines()

In [168]:
def seg_paraxysmal(data, af_start_loc, af_end_loc, duration, dataOverlap):
    labels = []
    tempBuf = []
    sig_start = 0
    
    for l in range(len(af_start_loc)):
        # Normal segment before AF
        for i in range(sig_start, af_start_loc[l], (duration - int(dataOverlap))):
            if i + duration > af_start_loc[l]:
                # Trim segment to end at AF start
                var = np.zeros(duration)
                valid_data_len = af_start_loc[l] - i
                var[:valid_data_len] = data[i:i + valid_data_len]
                tempBuf.append(var)
                labels.append(0)
                break
            else:
                var = np.zeros(duration)
                var[:len(data[i:i + duration])] = data[i:i + duration]
                tempBuf.append(var)
                labels.append(0)
        
        # AF segment handling
        af_seg_start = af_start_loc[l]
        while af_seg_start < af_end_loc[l]:
            var = np.zeros(duration)
            valid_data_len = min(duration, af_end_loc[l] - af_seg_start)
            var[:valid_data_len] = data[af_seg_start:af_seg_start + valid_data_len]
            tempBuf.append(var)
            labels.append(1)
            af_seg_start += (duration - int(dataOverlap))
        
        sig_start = af_end_loc[l]
    
    # Process remaining normal data after last AF episode
    for i in range(sig_start, len(data), (duration - int(dataOverlap))):
        var = np.zeros(duration)
        var[:len(data[i:i + duration])] = data[i:i + duration]
        tempBuf.append(var)
        labels.append(0)
                
    return tempBuf, labels


In [170]:
def af_start_end_loc(sample_path):
    #sample_path = os.path.join(DATA_PATH, test_set[2])
    #sig, _, fs = load_data(sample_path)
    sig, fields = wfdb.rdsamp(sample_path)
    ann_ref = wfdb.rdann(sample_path, 'atr')
    length = len(sig)
    sample_descrip = fields['comments']

    beat_loc = np.array(ann_ref.sample) # r-peak locations
    ann_note = np.array(ann_ref.aux_note) # rhythm change flag

    af_start_scripts = np.where((ann_note=='(AFIB') | (ann_note=='(AFL'))[0]
    af_end_scripts = np.where(ann_note=='(N')[0]
    af_start_loc=list()
    af_end_loc = list()
    for k in range(len(af_start_scripts)):
        af_start_loc.append(beat_loc[af_start_scripts[k]])
        af_end_loc.append(beat_loc[af_end_scripts[k]])
        
    return af_start_loc, af_end_loc

In [172]:
def downsample_signal(signal, target_length):
    """
    Downsample the signal to a target length.
    """
    downsampled_signal = resample(signal, target_length)
    print(f"Downsampled Signal Length: {len(downsampled_signal)} (Target: {target_length})")
    return downsampled_signal

In [ ]:
def process_and_save_images(data, labels, save_path, target_length=1000):

    # Ensure the save path exists
    os.makedirs(save_path, exist_ok=True)
    metadata_filename = os.path.join(save_path, "metadata.csv")

    # Check if labels are nested. If not, assume a single data segment.
    if not isinstance(labels[0], (list, np.ndarray)):
        # Single segment case
        print("Number of segments: 1")
        label = labels[0]
        with open(metadata_filename, mode='w', newline='') as file:
            writer = csv.writer(file)
            # Write header row in the CSV file
            writer.writerow(['Serial No', 'Data File Name', 'Segment No', 'Label'])

            # Process the single signal
            downsampled_signal = downsample_signal(data[0], target_length)

            # Normalize the signal
            min_val, max_val = np.min(downsampled_signal), np.max(downsampled_signal)
            if max_val - min_val == 0:
                print("Signal has constant value. Skipping GASF generation.")
                return  # Skip further processing

            normalized_signal = 2 * (downsampled_signal - min_val) / (max_val - min_val) - 1

            # Generate the GASF matrix
            phi = np.arccos(normalized_signal)
            gasf = np.sin(phi[:, None] - phi[None, :]).astype(np.float32)  
        


            # Save the GASF matrix as an image
            filename = f"sample1_segment1_label{label}.png"
            filepath = os.path.join(save_path, filename)
            print(f"Saving: {filepath}")
            plt.imsave(filepath, gasf, cmap='viridis')

            # Save metadata in the CSV file
            writer.writerow([1, filename, 1, label])
            
    else:
        # Multiple segments case (data is a list of samples, each with multiple segments)
        with open(metadata_filename, mode='w', newline='') as file:
            writer = csv.writer(file)
            # Write the header row in the CSV file
            writer.writerow(['Serial No', 'Data File Name', 'Segment No', 'Label'])

            # Process each sample and its segments
            for i, (buffered_segments, segment_labels) in enumerate(zip(data, labels)):
                print(f"\nProcessing sample {i + 1}/{len(data)}:")
                print(f"Number of segments in sample {i + 1}: {len(buffered_segments)}")
                for j, (signal, label) in enumerate(zip(buffered_segments, segment_labels)):
                    # Downsample the signal
                    downsampled_signal = downsample_signal(signal, target_length)

                    # Normalize the signal
                    min_val, max_val = np.min(downsampled_signal), np.max(downsampled_signal)
                    if max_val - min_val == 0:
                        print(f"Segment {j + 1} of sample {i + 1} has constant value. Skipping GASF generation.")
                        continue
                    normalized_signal = 2 * (downsampled_signal - min_val) / (max_val - min_val) - 1

                    # Generate the GASF matrix
                    phi = np.arccos(normalized_signal)
                    gasf = np.sin(phi[:, None] - phi[None, :]).astype(np.float32)

                    # Save the GASF matrix as an image
                    filename = f"sample{i+1}_segment{j+1}_label{label}.png"
                    filepath = os.path.join(save_path, filename)
                    print(f"Saving: {filepath}")
                    plt.imsave(filepath, gasf, cmap='viridis')

                    # Save metadata in the CSV file
                    writer.writerow([i + 1, filename, j + 1, label])

# Assuming the following helper functions are defined elsewhere in your code:
# load_data, RefInfo, af_start_end_loc, seg_paraxysmal, buffer, process_and_save_images

save_path = "D:\\research papers\\GASF"  # Modify as needed

# Initialize variables
sig_len = []  # To store the lengths of signals
labels_3 = []  # To store labels

for i in test_set:
    data = []   # To store segment(s) for the current sample
    labels = [] # To store corresponding labels for the current sample
    sample_path = os.path.join(DATA_PATH, i)
    sig, _, fs = load_data(sample_path)
    sig_len.append(sig.shape[0])
    d = RefInfo(sample_path)
    labels_3.append(d.class_true)
    
    # Initialize in case they are used
    af_start_loc = []
    af_end_loc = []
    
    if d.class_true == 2:
        af_start_loc, af_end_loc = af_start_end_loc(sample_path)
        if len(sig[:, 0]) < 6000:
            # If the entire signal is shorter than 6000, pad it
            padded_seg = np.pad(sig[:, 0], (0, 6000 - len(sig[:, 0])), 'constant')
            data.append(padded_seg)
            labels.append(1)
        else:
            # Segment the signal using the seg_paraxysmal function
            segments, lab = seg_paraxysmal(sig[:, 0], af_start_loc, af_end_loc, 6000, 3000)
            # Pad each segment if its length is less than 6000
            padded_segments = []
            for seg in segments:
                if len(seg) < 6000:
                    seg = np.pad(seg, (0, 6000 - len(seg)), 'constant')
                padded_segments.append(seg)
            # Convert list of segments to a 2D array (each row is a segment)
            data.append(np.vstack(padded_segments))
            labels.append(lab)
    else:
        if len(sig[:, 0]) < 6000:
            # If the signal is shorter than 6000, pad it
            padded_seg = np.pad(sig[:, 0], (0, 6000 - len(sig[:, 0])), 'constant')
            data.append(padded_seg)
            labels.append(d.class_true)
        else:
            # Segment the signal using the buffer function
            segments = buffer(sig[:, 0], 6000, 3000)
            padded_segments = []
            for seg in segments:
                if len(seg) < 6000:
                    seg = np.pad(seg, (0, 6000 - len(seg)), 'constant')
                padded_segments.append(seg)
            segments = np.array(padded_segments)
            lab = np.repeat(d.class_true, segments.shape[0])
            labels.append(lab)
            data.append(segments)

    # Create a directory for the current sample (using part of the path)
    sample_save_path = os.path.join(save_path, i.split('/')[1])
    os.makedirs(sample_save_path, exist_ok=True)
    
    # Process the data and save images (this function will generate GASF images, etc.)
    process_and_save_images(data, labels, sample_save_path)



Processing sample 1/1:
Number of segments in sample 1: 218
Downsampled Signal Length: 1000 (Target: 1000)
Saving: D:\research papers\Digital Twinning\GNM\data_68_2\sample1_segment1_label1.png
Downsampled Signal Length: 1000 (Target: 1000)
Saving: D:\research papers\Digital Twinning\GNM\data_68_2\sample1_segment2_label1.png
Downsampled Signal Length: 1000 (Target: 1000)
Saving: D:\research papers\Digital Twinning\GNM\data_68_2\sample1_segment3_label1.png
Downsampled Signal Length: 1000 (Target: 1000)
Saving: D:\research papers\Digital Twinning\GNM\data_68_2\sample1_segment4_label1.png
Downsampled Signal Length: 1000 (Target: 1000)
Saving: D:\research papers\Digital Twinning\GNM\data_68_2\sample1_segment5_label1.png
Downsampled Signal Length: 1000 (Target: 1000)
Saving: D:\research papers\Digital Twinning\GNM\data_68_2\sample1_segment6_label1.png
Downsampled Signal Length: 1000 (Target: 1000)
Saving: D:\research papers\Digital Twinning\GNM\data_68_2\sample1_segment7_label1.png
Downsampl